# Digital wind tunnel: 3D lattice Boltzmann around an STL airplane

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/lbm_wind_tunnel)

A small 3D wind tunnel in the style of the OpenLB library: the STL file of a model airplane is read into a triangle tensor, voxelised by ray casting over all lattice nodes at once, pitched to an angle of attack and placed in a D3Q19 lattice Boltzmann flow with TRT or MRT collisions, optional Smagorinsky LES and bounce-back on the body. Drag and lift come from momentum exchange; streamlines (batched RK4) and tracer particles show the flow. A sphere case checks the solver against a precomputed run of OpenLB itself and shows the residual.

**Vector form**

$$\partial_t\mathbf u + (\mathbf u\cdot\nabla)\mathbf u = -\tfrac1\rho\nabla p + \nabla\cdot\big[(\nu + \nu_t)(\nabla\mathbf u + \nabla\mathbf u^\mathsf T)\big],\qquad C_D = \frac{F_x}{\tfrac12\rho U^2 A},\quad C_L = \frac{F_z}{\tfrac12\rho U^2 A}$$

**Index (tensor) form**

$$f_i(\mathbf x + \mathbf e_i, t + 1) = f_i - \omega^+\Delta_i^+ - \omega^-\Delta_i^-,\quad \Delta_i^\pm = \tfrac12\big(\Delta_i \pm \Delta_{\bar\imath}\big),\ \Delta_i = f_i - f_i^\text{eq},\quad f^\text{eq}_i = w_i\rho\big(1 + 3e_{ia}u_a + \tfrac92(e_{ia}u_a)^2 - \tfrac32u_au_a\big),\quad F_a = 2\,L_{ixyz}\,f^*_{ixyz}\,e_{ia}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $f_{ixyz}$ | `f` | (19, Nx, Ny, Nz) | populations: fluid at node $(x, y, z)$ moving with lattice velocity $i$ |
| $e_{ia},\ w_i,\ \bar\imath$ | `lattice(dev)` | (19, 3), (19,), (19,) | D3Q19 velocities (rest, 6 faces, 12 edges), weights and opposite directions |
| $T_{tva}$ | `read_stl(path)` | (T, 3, 3) | STL triangles: triangle $t$, vertex $v$, coordinate $a$ |
| $s_{xyz}$ | `voxelise(tri, shape)` | (Nx, Ny, Nz) | inside if the signed crossings of a ray along $+x$ (winding number) are positive; all rays × all triangles, then searchsorted |
| $e_{ia}u_a$ | `einsum('ia,a...->i...', e, u)` | (19, Nx, Ny, Nz) | repeated $a$ ⇒ summed: velocity along each lattice direction |
| $\rho u_a = e_{ia}f_i$ | `einsum('ia,i...->a...', e, f)` | (3, Nx, Ny, Nz) | repeated $i$ ⇒ summed: momentum density |
| $M_{kj}\Delta_j$ | `einsum('kj,j...->k...', M, f - f_eq)` | (19, Nx, Ny, Nz) | MRT: non-equilibrium moments, each relaxed at its own rate $s_k$ |
| $\Pi_{ab} = e_{ia}e_{ib}\Delta_i$ | `einsum('ia,ib,i...->ab...', e, e, f - f_eq)` | (3, 3, Nx, Ny, Nz) | non-equilibrium momentum flux; its size sets the Smagorinsky eddy viscosity $\nu_t$ |
| $f(\mathbf x + \mathbf e_i)$ | `torch.roll(f[i], e_i)` | (Nx, Ny, Nz) | streaming: shift each population one link |
| $L_{ixyz}$ | `body_links(solid, e)` | (19, Nx, Ny, Nz) | 1 where a population leaves a fluid node towards the body (bounce-back as a masked swap) |
| $2\,L_{ixyz}f^*_{ixyz}e_{ia}$ | `2 * einsum('l,la->a', f_post[links], e[i])` | (3,) | momentum exchange: force on the body, summed over its boundary links |

Lattice units: node spacing and time step 1, free stream $U = 0.08$ (Mach 0.14), viscosity $\nu = UL/\mathrm{Re}$ with $L$ the body length in nodes, and $\tau = 3\nu + \tfrac12$. The airplane flies in an $80\times 36\times 26$ lattice with $L = 32$ (fuselage length; span 28 nodes), the optional NACA 0012 wing in $72\times 24\times 24$ (chord 20). Both STL files were generated for this problem (see data/SOURCES.md): a chunky model plane (fuselage of revolution, swept NACA 0018 wing, horizontal and vertical tail) whose parts overlap where they join, so the voxeliser counts winding numbers (signed crossings) instead of plain ray parity; and a wing that spans the periodic width, i.e. an infinite wing (compare $C_L$ with $2\pi\alpha$). Coefficients use the planform area (the body seen from above). Boundaries: periodic across the flow (a lattice of aircraft) and half-way bounce-back on the voxelised body; solve.simulate also offers a moving ground and a free-slip ceiling. Instead of an inlet and an outlet, the last 16 nodes are a fringe zone that relaxes the flow back to the free stream before it re-enters at $x = 0$; the speed ramps up over 300 steps (the early bump in $C_D$ is the pressure gradient that accelerates the air) and the animation starts after it. TRT uses the "magic" $\Lambda = 3/16$, MRT the D3Q19 rates of d'Humières et al. (2002), Smagorinsky $C_s = 0.17$. At the default Re = 1000 ($\tau \approx 0.508$) plain TRT blows up on this coarse lattice and LES keeps it stable: try switching LES off. Results at 8°: airplane $C_D \approx 0.39$, $C_L \approx 0.12$; wing $C_L \approx 0.31$ against $2\pi\alpha = 0.88$. The lift is low because the wings are only 1–3 cells thick, Re is laminar and the periodic images above and below act like a stack of biplanes: these are illustrations of the method, not predictions. Validation, run live after the airplane in the same execution (param validate, default on; or alone as body sphere_openlb) and shown in the residual panel: a sphere at Re = 20 ($D = 10$, $80\times 40\times 40$, BGK, equilibrium inlet) against OpenLB 1.7 on the same lattice, run once offline (data/openlb_reference_*.csv), 2000 steps: $C_D$ 3.360 vs 3.378 (−0.5 %); mid-plane $u_x$: relative L2 0.8 %, L∞ 1.1 %, max $|\Delta u|/U$ 1.3 %. The remaining difference comes from the boundary schemes (OpenLB uses interpolated velocity and pressure boundaries and keeps ringing with sound waves, ±5 % in $C_D$) and full-way versus half-way bounce-back; the collision (BGK) and the resolution are the same. Both codes sit above the Schiller–Naumann value 2.61 because this sphere is not free: OpenLB in a $160\times 60\times 60$ box gives 3.04 and with $D = 20$ in the same proportions 3.21. OpenLB served only as a precomputed reference; none of its (GPL) code is used here.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/lbm_wind_tunnel/main.py
"""Main loop: parameters → STL → voxels → lattice Boltzmann wind tunnel → drag and lift → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import drag_coefficient, schiller_naumann, viscosity_to_tau

DATA = Path(__file__).parent / "data"
U_IN, FRINGE, NOSE_X, RAMP = 0.08, 16, 12.0, 300  # free stream, fringe length, nose position, start-up ramp (steps)
VALIDATION_STEPS = 2000  # the sphere wake has settled by then (residuals as in test.py)
BODIES = {  # STL, lattice (Nx, Ny, Nz), body length in nodes; aircraft fly in a box periodic in y and z
    "airplane": {"stl": "airplane.stl", "shape": (80, 36, 26), "length": 32.0,
                 "reference": "no measurement: a generated model; C_L > 0 at positive angle of attack (planform area)"},
    "naca0012_wing": {"stl": "naca0012_wing.stl", "shape": (72, 24, 24), "length": 20.0,
                      "reference": "thin-aerofoil theory for an infinite wing: C_L ≈ 2π α (planform area)"},
    "sphere_openlb": {"stl": None, "shape": solve.SPHERE_CASE["shape"], "length": solve.SPHERE_CASE["diameter"],
                      "reference": "OpenLB 1.7, same lattice (data/openlb_reference_*.csv)"},
}  # fmt: skip


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    dt = torch.float32
    body, re, n_steps = params["body"], params["re"], params["steps"]
    collision, les = params["collision"], params["les"] == "on"
    cfg = BODIES[body]
    nx, ny, nz = cfg["shape"]
    check = body == "sphere_openlb"  # the OpenLB validation case: its own lattice, Re, collision and boundaries

    if check:
        re, collision, les, pitch = solve.SPHERE_CASE["re"], solve.SPHERE_CASE["collision"], False, 0.0
        tri = tensors.sphere_mesh(cfg["length"] / 2, solve.SPHERE_CASE["centre"], dtype=dt).to(dev)
        print(f"OpenLB check: Re = {re:.0f}, BGK, no LES, equilibrium inlet and copy outlet (other params ignored)")
    else:
        tri_stl = tensors.read_stl(DATA / cfg["stl"], dev, dt)  # (T, 3, 3) in mm
        pitch = params["pitch"]
        tri, _ = tensors.place_body(tri_stl, (nx, ny, nz), cfg["length"], NOSE_X, None, pitch_deg=pitch)  # centred
    solid = tensors.voxelise(tri, (nx, ny, nz))  # (Nx, Ny, Nz) bool: ray casting, winding number
    if check:
        area = torch.pi * cfg["length"] ** 2 / 4
    else:
        area = tensors.planform_area(solid)  # wing-style reference area: the body seen from above
    u_in = solve.SPHERE_CASE["u"] if check else U_IN
    nu = u_in * cfg["length"] / re
    tau = viscosity_to_tau(nu)

    e, w, _ = tensors.lattice(dev, dt)
    f = tensors.initial_populations(solid, u_in if check else 0.0, e, w)  # (19, Nx, Ny, Nz); tunnel: air at rest
    lo = (tri.reshape(-1, 3).min(0).values.cpu() - 2).clamp(min=0.5)
    hi = torch.minimum(tri.reshape(-1, 3).max(0).values.cpu() + 2, torch.tensor([nx, ny, nz]) - 1.5)
    ys = torch.linspace(float(lo[1]), float(hi[1]), 9)
    zs = torch.linspace(float(lo[2]), float(hi[2]), 5)
    x_rake = (float(tri[..., 0].min()) - 6) if check else NOSE_X - 6
    rake = torch.stack([torch.full((45,), x_rake), ys.repeat_interleave(5), zs.repeat(9)], 1).to(dev, dt)
    warmup = 0 if check else RAMP + 100  # the tunnel ramps the speed up smoothly; frames start once it is running
    result = solve.simulate(
        f, solid, u_in, tau, n_steps, collision=collision, les=les, walls=False, open_ends=check,
        fringe=0 if check else FRINGE, ramp=0 if check else RAMP,
        save_every=max(1, (n_steps - warmup) // 48), save_from=warmup, n_particles=2000,
        window=(lo.tolist(), hi.tolist()), seeds=rake,
    )  # fmt: skip
    coeffs = drag_coefficient(result["force"], u_in, area)  # (T, 3): C_D, C_side, C_L
    late = coeffs[n_steps // 2 :]
    cd, cl = late[:, 0].mean().item(), late[:, 2].mean().item()
    blockage = tensors.frontal_area(solid) / (ny * nz)

    print(f"lattice {nx}x{ny}x{nz} = {nx * ny * nz:,} nodes, D3Q19 {collision.upper()}{' + LES' if les else ''}")
    print(f"body '{body}': {int(solid.sum())} solid nodes from {len(tri)} STL triangles, length {cfg['length']:.0f},"
          f" pitch {pitch:.0f}°, reference area {area:.0f} nodes², blockage {blockage:.0%}")  # fmt: skip
    print(f"U = {u_in}, Re = U L / nu = {re:.0f}, nu = {nu:.5f}, tau = {tau:.4f}, Mach = {u_in * 3**0.5:.2f}")
    print(f"second half of the run: C_D = {cd:.3f} ± {late[:, 0].std().item():.3f}, C_L = {cl:+.3f}")
    if body == "naca0012_wing":
        print(f"2π α = {2 * torch.pi * torch.deg2rad(torch.tensor(pitch)).item():.3f}")
    if not check:
        print(f"reference: {cfg['reference']}; the lattice Re is far lower, so compare qualitatively")

    validation = None
    if check:  # this run is the validation case itself
        ux_final = solve.macroscopic(result["f"], e)[1][0, :, :, nz // 2].masked_fill(solid[:, :, nz // 2], 0.0)
        validation = solve.openlb_residuals(coeffs[:, 0], ux_final, DATA)
        validation["ux_ours"] = ux_final.cpu()
    elif params["validate"] == "on":  # option (b): run the OpenLB sphere case after the body, on its own lattice
        case = dict(solve.SPHERE_CASE, n_steps=VALIDATION_STEPS)
        sphere = solve.sphere_case(**case, device=dev)
        validation = solve.openlb_residuals(sphere["cd"], sphere["midplane_ux"], DATA)
        validation["ux_ours"] = sphere["midplane_ux"].cpu()
    if validation is not None:  # residual computed live against the bundled OpenLB CSVs
        validation["u"] = solve.SPHERE_CASE["u"]
        print(f"validation, tensor solver − OpenLB 1.7 (sphere, Re = 20, 80x40x40, BGK): C_D {validation['cd']:.3f}"
              f" vs {validation['cd_ref']:.3f} (ΔC_D {validation['cd_rel']:+.1%}); mid-plane u_x: rel. L2"
              f" {validation['l2']:.2%}, Linf {validation['linf']:.2%},"
              f" max |du|/U {validation['max_du']:.2%}")  # fmt: skip
        print(f"Schiller–Naumann (free sphere): C_D = {schiller_naumann(torch.tensor(20.0)).item():.3f}")
    pvd = plot.export_vtk(result, tri, solid, u_in, cfg["length"], out_dir)  # interactive 3D results
    size = sum(p.stat().st_size for p in pvd.parent.iterdir()) / 1e6
    print(f"interactive results: {pvd.relative_to(out_dir)} and its layers ({size:.1f} MB)")
    return plot.animate(result, coeffs, tri, solid, u_in, body, cfg["length"], out_dir / "lbm_wind_tunnel.gif",
                        validation=validation)  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/lbm_wind_tunnel/equation.py
r"""Physics → tensor translation.

Lattice Boltzmann in 3D (D3Q19). Populations f_{ixyz} carry the fluid at node (x, y, z) moving with lattice velocity
e_{ia} (i = 0..18, a ∈ {x, y, z}). The population index comes first, so each f[i] is one contiguous 3D grid.
Macroscopic fields are moments, sums over i:

    ρ = Σ_i f_i,        ρ u_a = Σ_i e_{ia} f_i

Each step relaxes f towards the local equilibrium (collision) and moves every population one link (streaming):

    f^{eq}_i = w_i ρ (1 + 3 e_{ia} u_a + 9/2 (e_{ia} u_a)² − 3/2 u_a u_a)
    f_i(x + e_i, t + 1) = f_i^*(x, t)

Collision operators, all in the same notation (Δ = f − f^eq):
  BGK:  f* = f − ω Δ                                         one rate, ν = (1/ω − ½)/3
  TRT:  f* = f − ω⁺ Δ⁺ − ω⁻ Δ⁻,  Δ^± = (Δ_i ± Δ_ī)/2         ī = opposite of i; Λ = (1/ω⁺ − ½)(1/ω⁻ − ½) = 3/16
  MRT:  f* = f − M⁻¹ S M Δ                                   M = 19 × 19 moment basis, S = diag(rates)

Smagorinsky LES raises the shear relaxation time locally from the non-equilibrium momentum flux
Π_{ab} = Σ_i e_{ia} e_{ib} Δ_i:  τ = ½ (τ₀ + √(τ₀² + 18√2 C_s² |Π| / ρ)).

Drag by momentum exchange: a population f*_i leaving a fluid node along a link into the body bounces back and hands
the body 2 f*_i e_i, so F_a = 2 Σ_{ixyz} L_{ixyz} f*_{ixyz} e_{ia}.
"""

import math

import torch


def density(f: torch.Tensor) -> torch.Tensor:
    r"""ρ = Σ_i f_i.   (19, ...) → (...)"""
    return torch.einsum("i...->...", f)


def momentum(f: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""ρu_a = Σ_i e_{ia} f_i  (i summed).   (19, ...), (19, 3) → (3, ...)"""
    return torch.einsum("ia,i...->a...", e, f)


def per_direction(v: torch.Tensor, like: torch.Tensor) -> torch.Tensor:
    """Reshape a per-direction vector (19,) so it broadcasts against a (19, ...) field."""
    return v.reshape(-1, *[1] * (like.dim() - 1))


def equilibrium(rho: torch.Tensor, u: torch.Tensor, e: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""f^{eq}_i = w_i ρ (1 + 3 e_{ia}u_a + 4.5 (e_{ia}u_a)² − 1.5 u_a u_a).   (...), (3, ...) → (19, ...)"""
    eu = torch.einsum("ia,a...->i...", e, u)  # e_ia u_a, a summed
    uu = torch.einsum("a...,a...->...", u, u)  # u_a u_a
    return per_direction(w, eu) * rho * (1 - 1.5 * uu + eu * (3 + 4.5 * eu))  # same polynomial, fewer temporaries


def collide_bgk(f: torch.Tensor, f_eq: torch.Tensor, omega) -> torch.Tensor:
    """BGK: relax every population at the same rate ω = 1/τ (a number or a field broadcastable to f[0])."""
    return f - omega * (f - f_eq)


def collide_trt(f: torch.Tensor, f_eq: torch.Tensor, omega_plus, omega_minus, opp: torch.Tensor) -> torch.Tensor:
    """TRT: the even part (Δ_i + Δ_ī)/2 relaxes with ω⁺ (sets the viscosity), the odd part with ω⁻.

    omega_plus / omega_minus are numbers or fields broadcastable to f[0] (local rates for LES).
    """
    d = f - f_eq
    d_opp = d[opp]  # Δ_ī: the same node, opposite direction
    # ω⁺ (Δ + Δ_ī)/2 + ω⁻ (Δ − Δ_ī)/2, regrouped by Δ and Δ_ī
    return f - 0.5 * (omega_plus + omega_minus) * d - 0.5 * (omega_plus - omega_minus) * d_opp


def collide_mrt(f: torch.Tensor, f_eq: torch.Tensor, m: torch.Tensor, rates: torch.Tensor) -> torch.Tensor:
    r"""MRT: go to moment space, relax each moment k at its own rate s_k, come back.

    f* = f − M⁻¹_{ik} s_k M_{kj} (f_j − f^eq_j)  (k, j summed). rates: (19,), or (19, ...) for local rates.
    """
    dm = torch.einsum("kj,j...->k...", m, f - f_eq)  # non-equilibrium moments
    s = per_direction(rates, dm) if rates.dim() == 1 else rates
    return f - torch.einsum("ik,k...->i...", torch.linalg.inv(m), s * dm)


def magic_omega_minus(omega_plus, magic: float = 3 / 16):
    """ω⁻ from Λ = (1/ω⁺ − ½)(1/ω⁻ − ½): Λ = 3/16 puts bounce-back walls exactly half-way between nodes."""
    return 1 / (magic / (1 / omega_plus - 0.5) + 0.5)


def smagorinsky_tau(f: torch.Tensor, f_eq: torch.Tensor, rho: torch.Tensor, e: torch.Tensor, tau0: float,
                    cs: float = 0.17) -> torch.Tensor:  # fmt: skip
    r"""Local relaxation time with a Smagorinsky eddy viscosity ν_t = (C_s Δx)² |S|.

    Π_ab = Σ_i e_ia e_ib (f_i − f^eq_i)  (i summed);  τ = ½ (τ₀ + √(τ₀² + 18√2 C_s² √(Π_ab Π_ab) / ρ)).
    """
    pi = torch.einsum("ia,ib,i...->ab...", e, e, f - f_eq)  # (3, 3, ...)
    norm = torch.sqrt(torch.einsum("ab...,ab...->...", pi, pi))  # |Π|
    return 0.5 * (tau0 + torch.sqrt(tau0**2 + 18 * math.sqrt(2) * cs**2 * norm / rho))


def stream(f: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    """Move every population one node along its own velocity e_i (periodic wrap-around via torch.roll)."""
    return torch.stack([torch.roll(f[i], shifts=(ex, ey, ez), dims=(0, 1, 2))
                        for i, (ex, ey, ez) in enumerate(e.int().tolist())])  # fmt: skip


def bounce_back(f_streamed: torch.Tensor, f_post: torch.Tensor, incoming: tuple, opp: torch.Tensor) -> torch.Tensor:
    """Half-way bounce-back as a masked swap: where population i would have come out of the body (the entries of the
    mask incoming[i, x, y, z], given as its index tuple), put the population that left x towards the body, reversed:
    f_i(x) = f*_ī(x)."""
    i, x, y, z = incoming
    f_streamed[i, x, y, z] = f_post[opp[i], x, y, z]
    return f_streamed


def momentum_exchange_force(f_post: torch.Tensor, links: tuple, e: torch.Tensor) -> torch.Tensor:
    r"""F_a = 2 Σ_{ixyz} L_{ixyz} f*_{ixyz} e_{ia}  (i, x, y, z summed): force on the body (lattice units).   → (3,)

    links: index tuple of the boundary-link mask L, so only those few populations are gathered."""
    i, x, y, z = links
    return 2 * torch.einsum("l,la->a", f_post[i, x, y, z], e[i])


def drag_coefficient(force, u: float, area: float):
    """C = F / (½ ρ U² A) with ρ = 1."""
    return force / (0.5 * u**2 * area)


def schiller_naumann(re: torch.Tensor) -> torch.Tensor:
    """Sphere drag correlation (Schiller & Naumann 1933), a few % for Re < 800: C_D = 24/Re (1 + 0.15 Re^0.687)."""
    return 24 / re * (1 + 0.15 * re**0.687)


def viscosity_to_tau(nu: float) -> float:
    """Relaxation time from the lattice viscosity: ν = (τ − ½)/3."""
    return 3 * nu + 0.5


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/lbm_wind_tunnel/tensors.py
"""Build the concrete tensors: the D3Q19 lattice and its moment basis, the STL triangles, the voxelised body and the
links where populations bounce off it."""

import struct
from pathlib import Path

import numpy as np
import torch


def lattice(device=None, dtype=torch.float32) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """D3Q19: velocities e_ia (19, 3), weights w_i (19,) and the opposite direction opp[i] (19,).

    i = 0 is rest, 1-6 the six faces (weight 1/18), 7-18 the twelve edges (weight 1/36).
    """
    faces = [[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]]
    edges = [[a, b, 0] for a in (1, -1) for b in (1, -1)] + [[a, 0, b] for a in (1, -1) for b in (1, -1)]
    edges += [[0, a, b] for a in (1, -1) for b in (1, -1)]
    e = torch.tensor([[0, 0, 0]] + faces + edges, device=device, dtype=dtype)
    w = torch.tensor([1 / 3] + [1 / 18] * 6 + [1 / 36] * 12, device=device, dtype=dtype)
    opp = (e[:, None, :] == -e[None, :, :]).all(-1).int().argmax(1)  # opp[i]: e_opp = -e_i
    return e, w, opp


def moment_basis(e: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """MRT moment matrix M (19, 19) and a mask of odd moments (19,).

    Rows are the polynomials of d'Humières et al. (2002) — density, momentum, energy, stresses, heat fluxes and
    higher moments — evaluated on the lattice velocities and made orthogonal by Gram–Schmidt.
    """
    x, y, z = e.T
    r2 = x * x + y * y + z * z
    polys = [
        (torch.ones_like(x), 0), (x, 1), (y, 1), (z, 1),  # density, momentum (conserved)
        (r2, 2), (2 * x * x - y * y - z * z, 2), (y * y - z * z, 2), (x * y, 2), (y * z, 2), (x * z, 2),  # stresses
        (x * r2, 3), (y * r2, 3), (z * r2, 3), (x * (y * y - z * z), 3), (y * (z * z - x * x), 3),
        (z * (x * x - y * y), 3), (r2 * r2, 4), ((2 * x * x - y * y - z * z) * r2, 4), ((y * y - z * z) * r2, 4),
    ]  # fmt: skip
    rows = []
    for p, _ in polys:  # Gram–Schmidt (plain sum over i as inner product)
        for q in rows:
            p = p - (p @ q) / (q @ q) * q
        rows.append(p)
    odd = torch.tensor([deg % 2 == 1 for _, deg in polys], device=e.device)
    return torch.stack(rows), odd


# ---------------------------------------------------------------- geometry: STL → triangles → voxels


def read_stl(path: Path, device=None, dtype=torch.float32) -> torch.Tensor:
    """Triangles (T, 3, 3) [triangle, vertex, xyz] from a binary or ASCII STL file."""
    raw = Path(path).read_bytes()
    n = struct.unpack("<I", raw[80:84])[0] if len(raw) >= 84 else 0
    if len(raw) == 84 + 50 * n:  # binary: 80-byte header, count, then 50 bytes per triangle
        rec = np.frombuffer(
            raw, dtype=np.dtype([("n", "<f4", 3), ("v", "<f4", (3, 3)), ("a", "<u2")]), count=n, offset=84
        )
        tri = rec["v"].astype(np.float64)
    else:  # ASCII: every "vertex x y z" line
        words = raw.decode("ascii", "replace").split()
        idx = [k for k, word in enumerate(words) if word == "vertex"]
        tri = np.array([[float(words[k + 1]), float(words[k + 2]), float(words[k + 3])] for k in idx]).reshape(-1, 3, 3)
    return torch.tensor(tri, device=device, dtype=dtype)


def mesh_volume(tri: torch.Tensor) -> torch.Tensor:
    """V = Σ_t v0 · (v1 × v2) / 6 (divergence theorem; positive for outward-facing triangles)."""
    return torch.einsum("ta,ta->", tri[:, 0], torch.linalg.cross(tri[:, 1], tri[:, 2])) / 6


def edge_use_counts(tri: torch.Tensor, decimals: int = 4) -> torch.Tensor:
    """How many triangles share each edge (2 everywhere ⇔ closed surface)."""
    verts, ids = torch.unique(torch.round(tri.reshape(-1, 3) * 10**decimals), dim=0, return_inverse=True)
    ids = ids.reshape(-1, 3)
    edges = torch.cat([ids[:, [0, 1]], ids[:, [1, 2]], ids[:, [2, 0]]])  # (3T, 2)
    edges = torch.sort(edges, dim=1).values
    return torch.unique(edges, dim=0, return_counts=True)[1]


def voxelise(tri: torch.Tensor, shape: tuple[int, int, int], max_pairs: int = 4_000_000) -> torch.Tensor:
    """Solid mask (Nx, Ny, Nz) by ray casting along +x from every node, all rays and all triangles at once (chunked
    over rays to bound memory).

    For the ray through (y, z) and triangle (a, b, c): barycentric weights in the (y, z) projection decide whether it
    hits, and x_hit = a_x + λ_b (b_x − a_x) + λ_c (c_x − a_x). Each hit counts +1 if the outward normal points along
    +x (the ray leaves a solid) and −1 otherwise; the sum over the hits ahead of a node is its winding number. For one
    closed surface this is the ray-parity test (inside ⇔ odd number of crossings ⇔ winding 1); for parts that overlap
    (wings through a fuselage) it stays right where plain parity would cancel. Hits ahead come from a searchsorted.
    """
    nx, ny, nz = shape
    dev, dt = tri.device, tri.dtype
    yy, zz = torch.meshgrid(
        torch.arange(ny, device=dev, dtype=dt), torch.arange(nz, device=dev, dtype=dt), indexing="ij"
    )
    rays = torch.stack([yy.reshape(-1) + 1.234e-5, zz.reshape(-1) + 2.345e-5], 1)  # (R, 2), off-grid to avoid edges
    a, b, c = tri[:, 0], tri[:, 1], tri[:, 2]  # (T, 3)
    ab, ac = (b - a)[:, 1:], (c - a)[:, 1:]  # (T, 2) in the (y, z) plane

    def cross(u, v):
        return u[..., 0] * v[..., 1] - u[..., 1] * v[..., 0]

    det = cross(ab, ac)  # (T,) = x-component of the (unnormalised) triangle normal
    xs = torch.arange(nx, device=dev, dtype=dt)
    inside = []
    chunk = max(1, max_pairs // max(1, len(tri)))
    for start in range(0, len(rays), chunk):  # chunks of rays, not single nodes
        ap = rays[start : start + chunk, None, :] - a[None, :, 1:]  # (r, T, 2)
        lb = cross(ap, ac) / det
        lc = cross(ab, ap) / det
        hit = (lb >= 0) & (lc >= 0) & (lb + lc <= 1) & (det != 0)
        x_hit = a[:, 0] + lb * (b - a)[:, 0] + lc * (c - a)[:, 0]  # (r, T)
        x_hit, order = torch.where(hit, x_hit, torch.full_like(x_hit, float("inf"))).sort(dim=1)
        sign = torch.gather(torch.where(hit, torch.sign(det).expand_as(hit), torch.zeros_like(x_hit)), 1, order)
        before = torch.nn.functional.pad(torch.cumsum(sign, 1), (1, 0))  # (r, T+1): signed hits up to each index
        behind = torch.searchsorted(x_hit, xs.expand(len(x_hit), nx).contiguous())  # hits with x_hit < x
        winding = before[:, -1:] - torch.gather(before, 1, behind)  # signed hits ahead of each node
        inside.append(winding > 0.5)  # (r, Nx)
    return torch.cat(inside).T.reshape(nx, ny, nz)


def place_body(tri: torch.Tensor, shape: tuple[int, int, int], length: float, nose_x: float, clearance: float | None,
               pitch_deg: float = 0.0) -> tuple[torch.Tensor, float]:  # fmt: skip
    """Scale an STL (any unit) so its x-extent is `length` lattice spacings, pitch it nose-up by pitch_deg about its
    centre, put the nose at x = nose_x, centre it in y, and lift it `clearance` (STL units) above the ground plane
    (the half-way wall at z = −½), or centre it in z if clearance is None (free flight). Returns lattice-coordinate
    triangles and the scale (lattice units per STL unit)."""
    v = tri.reshape(-1, 3)
    scale = length / float(v[:, 0].max() - v[:, 0].min())
    v = (v - v.mean(0)) * scale
    th = torch.deg2rad(torch.tensor(pitch_deg, dtype=v.dtype, device=v.device))
    rot = torch.stack([torch.stack([torch.cos(th), torch.zeros_like(th), torch.sin(th)]),
                       torch.tensor([0.0, 1.0, 0.0], dtype=v.dtype, device=v.device),
                       torch.stack([-torch.sin(th), torch.zeros_like(th), torch.cos(th)])])  # fmt: skip
    v = torch.einsum("ab,nb->na", rot, v)  # nose up for positive pitch (x forward, z up)
    lo = v.min(0).values
    shift = torch.tensor([nose_x, 0.0, -0.5 + (clearance or 0.0) * scale], dtype=v.dtype, device=v.device) - lo
    shift[1] = (shape[1] - 1) / 2 - (v[:, 1].max() + v[:, 1].min()) / 2
    if clearance is None:
        shift[2] = (shape[2] - 1) / 2 - (v[:, 2].max() + v[:, 2].min()) / 2
    return (v + shift).reshape(-1, 3, 3), scale


def sphere_mesh(radius: float, centre, n_theta: int = 24, n_phi: int = 48, dtype=torch.float32) -> torch.Tensor:
    """UV sphere (T, 3, 3), outward-facing triangles."""
    th = torch.linspace(0, torch.pi, n_theta + 1, dtype=dtype)
    ph = torch.linspace(0, 2 * torch.pi, n_phi + 1, dtype=dtype)
    t, p = torch.meshgrid(th, ph, indexing="ij")
    pts = torch.stack([torch.sin(t) * torch.cos(p), torch.sin(t) * torch.sin(p), torch.cos(t)], -1) * radius
    pts = pts + torch.tensor(centre, dtype=dtype)
    a, b, c, d = pts[:-1, :-1], pts[1:, :-1], pts[1:, 1:], pts[:-1, 1:]  # quad corners (n_theta, n_phi, 3)
    tri = torch.cat([torch.stack([a, b, c], 2).reshape(-1, 3, 3), torch.stack([a, c, d], 2).reshape(-1, 3, 3)])
    area = torch.linalg.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]).norm(dim=1)
    return tri[area > 1e-12 * radius**2]  # drop the degenerate triangles at the poles


def box_mesh(lo, hi, dtype=torch.float32) -> torch.Tensor:
    """Axis-aligned box (12, 3, 3), outward-facing triangles."""
    lo, hi = torch.tensor(lo, dtype=dtype), torch.tensor(hi, dtype=dtype)
    corner = torch.stack([torch.where(torch.tensor([(k >> j) & 1 for j in range(3)]).bool(), hi, lo) for k in range(8)])
    quads = [[0, 2, 3, 1], [4, 5, 7, 6], [0, 1, 5, 4], [2, 6, 7, 3], [0, 4, 6, 2], [1, 3, 7, 5]]  # outward order
    return torch.stack([corner[[q[0], q[k], q[k + 1]]] for q in quads for k in (1, 2)])


# ---------------------------------------------------------------- lattice masks and initial state


def body_links(solid: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    """links[i, x, y, z] = True where node x is fluid and x + e_i is inside the body (a population leaving along e_i
    hits the body and bounces back).   (19, Nx, Ny, Nz)"""
    hits = [torch.roll(solid, shifts=(-ex, -ey, -ez), dims=(0, 1, 2)) for ex, ey, ez in e.int().tolist()]
    return ~solid & torch.stack(hits)


def frontal_area(solid: torch.Tensor) -> float:
    """Projected area seen by the flow: number of (y, z) columns that contain a solid node."""
    return float(solid.any(dim=0).sum())


def planform_area(solid: torch.Tensor) -> float:
    """Area seen from above (wings): number of (x, y) columns that contain a solid node."""
    return float(solid.any(dim=2).sum())


def initial_populations(solid: torch.Tensor, u_in: float, e: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """Equilibrium (19, Nx, Ny, Nz) at ρ = 1 with the free-stream velocity outside the body, at rest inside."""
    from .equation import equilibrium

    u = torch.zeros(3, *solid.shape, device=e.device, dtype=e.dtype)
    u[0] = u_in
    u[:, solid] = 0
    return equilibrium(torch.ones(solid.shape, device=e.device, dtype=e.dtype), u, e, w)


def read_reference_csv(path: Path) -> dict[str, torch.Tensor]:
    """Columns of a CSV with '#' comment lines and a header row, as float64 tensors."""
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln and not ln.startswith("#")]
    names = lines[0].split(",")
    data = np.array([[float(v) for v in ln.split(",")] for ln in lines[1:]])
    return {name: torch.tensor(data[:, k], dtype=torch.float64) for k, name in enumerate(names)}


def surface_nets(field: torch.Tensor, level: float, periodic: bool = False) -> tuple[torch.Tensor, torch.Tensor]:
    """Iso-surface {field = level} by naive surface nets, for all cells at once.

    field (Nx, Ny, Nz) on integer grid points. Every cell (cube of 8 neighbouring points) whose corners straddle the
    level gets one vertex: the mean of the points where the level crosses its 12 edges (linear interpolation along
    each edge). Every grid edge that crosses the level gets a quad joining the 4 cells around it, split into two
    triangles oriented away from the inside (field > level). periodic=True wraps all three directions (a torus).
    Returns vertices (V, 3) in grid units and triangles (F, 3) of vertex indices."""
    dims = torch.tensor(field.shape, device=field.device)
    cells = tuple(int(n) if periodic else int(n) - 1 for n in field.shape)

    def at(offset):  # field at cell corner (i, j, k) + offset, for every cell
        f = torch.roll(field, shifts=tuple(-int(o) for o in offset), dims=(0, 1, 2)) if any(offset) else field
        return f[: cells[0], : cells[1], : cells[2]]

    corners = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]
    values = {c: at(c) - level for c in corners}
    edges = [(p, q) for p in corners for q in corners if p < q and sum(abs(x - y) for x, y in zip(p, q)) == 1]
    total = torch.zeros(*cells, 3, dtype=field.dtype, device=field.device)
    count = torch.zeros(cells, dtype=field.dtype, device=field.device)
    for p, q in edges:  # the 12 edges of a cell (a loop over edge types, not over cells)
        a, b = values[p], values[q]
        cross = (a > 0) != (b > 0)
        t = torch.where(cross, a / (a - b).masked_fill(~cross, 1.0), torch.zeros_like(a))
        point = torch.tensor(p, dtype=field.dtype, device=field.device) + t[..., None] * torch.tensor(
            [qq - pp for pp, qq in zip(p, q)], dtype=field.dtype, device=field.device)  # fmt: skip
        total += point * cross[..., None]
        count += cross
    active = count > 0
    grid = torch.stack(torch.meshgrid(*[torch.arange(n, device=field.device) for n in cells], indexing="ij"), -1)
    verts = (grid + total / count.clamp(min=1)[..., None])[active]  # (V, 3)
    index = torch.full(cells, -1, dtype=torch.long, device=field.device)
    index[active] = torch.arange(int(active.sum()), device=field.device)

    inside = field > level
    faces = []
    for axis in range(3):  # grid edges along each axis
        b_ax, c_ax = [d for d in range(3) if d != axis]
        step = [0, 0, 0]
        step[axis] = 1
        nxt = torch.roll(inside, shifts=-1, dims=axis)
        crossing = inside != nxt
        if not periodic:  # the edge and all four cells around it must lie inside the box
            valid = torch.zeros_like(crossing)
            sl = [slice(None)] * 3
            sl[axis] = slice(0, int(dims[axis]) - 1)
            sl[b_ax] = slice(1, int(dims[b_ax]) - 1)
            sl[c_ax] = slice(1, int(dims[c_ax]) - 1)
            valid[tuple(sl)] = True
            crossing &= valid
        p = crossing.nonzero()  # (E, 3) edge start points
        flip = ~inside[p[:, 0], p[:, 1], p[:, 2]] ^ (axis == 1)  # (axis, b, c) is left-handed for the y axis

        around = []  # vertex index of the cells (p − db e_b − dc e_c) sharing this edge
        for db, dc in ((0, 0), (1, 0), (1, 1), (0, 1)):
            q = p.clone()
            q[:, b_ax] -= db
            q[:, c_ax] -= dc
            q %= torch.tensor(cells, device=field.device)
            around.append(index[q[:, 0], q[:, 1], q[:, 2]])
        v00, v10, v11, v01 = around
        quad = torch.stack([v00, v10, v11, v01], 1)
        quad = torch.where(flip[:, None], quad.flip(1), quad)
        faces.append(torch.cat([quad[:, [0, 1, 2]], quad[:, [0, 2, 3]]]))
    faces = torch.cat(faces)
    return verts, faces[(faces >= 0).all(1)]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/lbm_wind_tunnel/solve.py
"""The wind-tunnel loop: collide (BGK / TRT / MRT, optional Smagorinsky LES) → stream → bounce-back on the body →
moving ground and free-slip ceiling → inlet/outlet or a fringe zone that restores the free stream; plus tracer
particles carried by the flow."""

import math

import torch
import torch.nn.functional as F

from problems._common.progress import steps

from . import tensors
from .equation import (
    bounce_back,
    collide_bgk,
    collide_mrt,
    collide_trt,
    density,
    drag_coefficient,
    equilibrium,
    magic_omega_minus,
    momentum,
    momentum_exchange_force,
    smagorinsky_tau,
    stream,
)

# d'Humières et al. (2002) rates for the non-hydrodynamic moments of D3Q19, in the order of tensors.moment_basis;
# the five shear-stress moments (rows 5-9) relax with 1/τ and set the viscosity.
MRT_RATES = [0.0, 0.0, 0.0, 0.0, 1.19, None, None, None, None, None, 1.2, 1.2, 1.2, 1.98, 1.98, 1.98, 1.4, 1.4, 1.4]
SHEAR = slice(5, 10)


def macroscopic(f: torch.Tensor, e: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """ρ (Nx, Ny, Nz) and u (3, Nx, Ny, Nz) from the populations."""
    rho = density(f)
    return rho, momentum(f, e) / rho


def collide(f, f_eq, rho, e, opp, tau0, collision, les, cs, m=None):
    """One collision with the chosen operator; LES makes the shear relaxation time a field τ(x)."""
    tau = smagorinsky_tau(f, f_eq, rho, e, tau0, cs) if les else torch.tensor(tau0, dtype=f.dtype, device=f.device)
    omega = 1 / tau
    if collision == "bgk":
        return collide_bgk(f, f_eq, omega)
    if collision == "trt":
        return collide_trt(f, f_eq, omega, magic_omega_minus(omega), opp)
    rates = torch.tensor([0.0 if r is None else r for r in MRT_RATES], dtype=f.dtype, device=f.device)
    rates = rates.reshape(19, *[1] * omega.dim()).expand(19, *omega.shape).clone()
    rates[SHEAR] = omega
    return collide_mrt(f, f_eq, m, rates)


def interpolate(u: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Trilinear velocity at particle positions: u (3, Nx, Ny, Nz), p (P, 3) in node coordinates → (P, 3)."""
    size = torch.tensor(u.shape[1:], dtype=p.dtype, device=p.device)
    grid = (2 * p / (size - 1) - 1).flip(-1)  # grid_sample wants (W, H, D) = (z, y, x) order in [-1, 1]
    out = F.grid_sample(u[None], grid[None, None, None], align_corners=True)  # (1, 3, 1, 1, P)
    return out[0, :, 0, 0].T


def streamlines(u: torch.Tensor, seeds: torch.Tensor, solid: torch.Tensor, n_points: int, h: float, exit_x: float):
    """Integrate dx/ds = u(x) from all seeds at once with RK4 (trilinear u). Points past exit_x, outside the box or
    inside the body become NaN (the line stops). Returns lines (S, K+1, 3) and speeds |u| along them (S, K+1)."""
    size = torch.tensor(u.shape[1:], dtype=seeds.dtype, device=seeds.device)
    p, points = seeds, [seeds]
    for _ in range(n_points):  # integration steps; every seed advances together
        k1 = interpolate(u, p)
        k2 = interpolate(u, p + 0.5 * h * k1)
        k3 = interpolate(u, p + 0.5 * h * k2)
        k4 = interpolate(u, p + h * k3)
        p = torch.minimum(torch.clamp(p + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4), min=0), size - 1)
        points.append(p)
    line = torch.stack(points, 1)  # (S, K+1, 3)
    speed = interpolate(u, line.reshape(-1, 3)).norm(dim=-1).reshape(line.shape[:2])
    idx = line.round().long()
    stop = (line[..., 0] > exit_x) | solid[idx[..., 0], idx[..., 1], idx[..., 2]]
    stop = torch.cummax(stop.int(), dim=1).values.bool()  # once stopped, stays stopped
    return line.masked_fill(stop[..., None], float("nan")), speed.masked_fill(stop, float("nan"))


def simulate(
    f: torch.Tensor, solid: torch.Tensor, u_in: float, tau: float, n_steps: int, collision: str = "trt",
    les: bool = False, cs: float = 0.17, walls: bool = True, open_ends: bool = True, save_every: int = 0,
    n_particles: int = 0, window=None, seed: int = 0, ramp: int = 0, fringe: int = 0, seeds=None, save_from: int = 0,
) -> dict:  # fmt: skip
    """Run n_steps lattice updates on populations f (19, Nx, Ny, Nz).

    walls=True: moving ground (speed u_in) under z = 0 and a free-slip ceiling above z = Nz−1; otherwise periodic in z.
    open_ends=True: equilibrium inlet at x = 0 and zero-gradient outlet at x = Nx−1; otherwise periodic in x.
    fringe > 0 (use with open_ends=False): over the last `fringe` nodes the populations are relaxed towards the free
    stream f_eq(1, U), so the periodic box becomes a wind tunnel: the wake is wiped out and uniform flow re-enters at
    x = 0, and the sound waves of the start are absorbed instead of bouncing between an inlet and an outlet.
    ramp > 0: the free-stream and ground speed rise smoothly from 0 to u_in over `ramp` steps (no impulsive start).
    y is always periodic. Returns the force history (n_steps, 3), final populations, and (if save_every, from step
    save_from on) snapshots of the speed in the mid-span x–z plane, of the tracer particles (released continuously
    near x = 1 inside `window`, recycled when they leave) and of streamlines from the rake `seeds` (S, 3).
    """
    dev, dt = f.device, f.dtype
    _, nx, ny, nz = f.shape
    e, w, opp = tensors.lattice(dev, dt)
    m = tensors.moment_basis(e)[0] if collision == "mrt" else None
    links_mask = tensors.body_links(solid, e)  # (19, Nx, Ny, Nz)
    links = links_mask.nonzero(as_tuple=True)  # index tuple of the few boundary links
    incoming = links_mask[opp].nonzero(as_tuple=True)  # population i at x would come out of the body
    frozen = f[:, solid]  # nodes inside the body keep their (rest) state
    up = (e[:, 2] > 0).nonzero().squeeze(1)  # directions entering the fluid from the ground
    down = (e[:, 2] < 0).nonzero().squeeze(1)  # directions entering from the ceiling
    mirror = [
        int(((e == e[i] * torch.tensor([1, 1, -1], dtype=dt, device=dev)).all(1)).nonzero()) for i in down.tolist()
    ]
    wall_kick = (6 * w[up] * e[up, 0])[:, None, None]  # moving wall: 2 w_i ρ (e_i·u_w)/c_s² per unit speed, ρ ≈ 1
    ones = torch.ones(ny, nz, device=dev, dtype=dt)

    def free_stream(speed: float) -> torch.Tensor:
        """f_eq(ρ = 1, u = (speed, 0, 0)) on a y–z plane: (19, Ny, Nz)."""
        return equilibrium(
            ones, torch.tensor([speed, 0.0, 0.0], device=dev, dtype=dt)[:, None, None].expand(3, ny, nz), e, w
        )

    x = torch.arange(fringe, device=dev, dtype=dt)
    sigma = (0.3 * torch.sin(torch.pi * (x + 1) / (fringe + 1)) ** 2)[:, None, None]  # (fringe, 1, 1): smooth bump
    exit_x = nx - 3 - fringe  # tracer particles are recycled here

    gen = torch.Generator(device="cpu").manual_seed(seed)
    lo, hi = (torch.tensor(window[0], dtype=dt), torch.tensor(window[1], dtype=dt)) if window else (None, None)

    def spawn(k: int, anywhere: bool) -> torch.Tensor:
        r = torch.rand(k, 3, generator=gen, dtype=dt)
        p = lo + r * (hi - lo)
        p[:, 0] = 1 + r[:, 0] * ((exit_x - 1) if anywhere else 1.5)
        return p.to(dev)

    particles = spawn(n_particles, True) if n_particles else None
    age = torch.zeros(n_particles, device=dev, dtype=dt)  # lattice steps since each particle was (re)released
    forces, slices, tracks, speeds, saved, lines, line_speeds, ux_xy, ages = [], [], [], [], [], [], [], [], []
    for n in steps(n_steps, "lattice steps"):
        speed_now = u_in * math.sin(0.5 * math.pi * min(1.0, n / ramp)) ** 2 if ramp else u_in
        rho, u = macroscopic(f, e)
        f_eq = equilibrium(rho, u, e, w)
        f_post = collide(f, f_eq, rho, e, opp, tau, collision, les, cs, m)
        if fringe:  # f* ← f* − σ(x) (f* − f_eq(1, U)) in the last nodes: the free stream is restored before x wraps
            tail = f_post[:, nx - fringe :]
            f_post[:, nx - fringe :] = tail - sigma * (tail - free_stream(speed_now)[:, None])
        forces.append(momentum_exchange_force(f_post, links, e))  # populations about to hit the body
        f = bounce_back(stream(f_post, e), f_post, incoming, opp)
        f[:, solid] = frozen
        if walls:
            f[down, :, :, -1] = f[mirror, :, :, 0]  # specular reflection: arrived through the wrap = mirrored copy
            f[up, :, :, 0] = f_post[opp[up], :, :, 0] + wall_kick * speed_now  # moving ground: bounce back + push
        if open_ends:
            f[:, 0] = free_stream(u_in)  # equilibrium inlet
            f[:, -1] = f[:, -2]  # zero-gradient outlet
        if n_particles:
            particles = particles + interpolate(u, particles)  # dx = u dt with dt = 1
            age += 1
            idx = particles.round().long()
            idx = torch.minimum(idx.clamp(min=0), torch.tensor([nx - 1, ny - 1, nz - 1], device=dev))
            gone = (particles[:, 0] > exit_x) | solid[idx[:, 0], idx[:, 1], idx[:, 2]] | (particles[:, 1:] < 0).any(1)
            gone |= (particles[:, 1] > ny - 1) | (particles[:, 2] > nz - 1)
            if gone.any():
                particles[gone] = spawn(int(gone.sum()), False)
                age[gone] = 0
        if save_every and n >= save_from and (n - save_from) % save_every == 0:
            saved.append(n)
            speed = u.norm(dim=0).masked_fill(solid, float("nan"))
            slices.append(speed[:, ny // 2].clone())  # mid-span x–z plane
            ux_xy.append(u[0, :, :, nz // 2].masked_fill(solid[:, :, nz // 2], 0.0))  # mid-height x–y plane, u_x
            if n_particles:
                tracks.append(particles.clone())
                ages.append(age.clone())
                speeds.append(interpolate(u, particles).norm(dim=-1))
            if seeds is not None:
                line, line_speed = streamlines(u, seeds, solid, n_points=int(1.1 * exit_x), h=1 / u_in, exit_x=exit_x)
                lines.append(line)
                line_speeds.append(line_speed)
    out = {"force": torch.stack(forces), "f": f, "steps": torch.tensor(saved)}
    if save_every:
        out["slices"], out["ux_xy"] = torch.stack(slices), torch.stack(ux_xy)
        if n_particles:
            out["tracks"], out["speeds"], out["ages"] = torch.stack(tracks), torch.stack(speeds), torch.stack(ages)
        if seeds is not None:
            out["lines"], out["line_speeds"] = torch.stack(lines), torch.stack(line_speeds)
    return out


# ---------------------------------------------------------------- validation: a sphere at Re = 20

SPHERE_CASE = {"shape": (80, 40, 40), "diameter": 10.0, "centre": (20.0, 19.5, 19.5), "re": 20.0, "u": 0.05,
               "n_steps": 2000, "collision": "bgk"}  # fmt: skip


def sphere_case(shape, diameter, centre, re, u, n_steps, collision, device=None, dtype=torch.float32) -> dict:
    """Uniform flow past a sphere in a box periodic across the flow (the OpenLB reference case)."""
    e, w, _ = tensors.lattice(device, dtype)
    nx, ny, nz = shape
    grid = torch.stack(torch.meshgrid(*[torch.arange(s, device=device, dtype=dtype) for s in shape], indexing="ij"), -1)
    solid = ((grid - torch.tensor(centre, device=device, dtype=dtype)) ** 2).sum(-1) <= (diameter / 2) ** 2
    tau = 0.5 + 3 * u * diameter / re
    f = tensors.initial_populations(solid, u, e, w)
    out = simulate(f, solid, u, tau, n_steps, collision=collision, walls=False, open_ends=True)
    cd = drag_coefficient(out["force"][:, 0], u, math.pi * diameter**2 / 4)
    _, vel = macroscopic(out["f"], e)
    return {"cd": cd, "cd_mean": cd[int(0.8 * n_steps) :].mean().item(),
            "midplane_ux": vel[0, :, :, nz // 2].masked_fill(solid[:, :, nz // 2], 0.0).double()}  # fmt: skip


def openlb_residuals(cd: torch.Tensor, ux_plane: torch.Tensor, data_dir) -> dict:
    """Our sphere run minus the precomputed OpenLB run (data/openlb_reference_*.csv): relative C_D difference
    (time averages over the last 40 % of each run) and the relative L2 / L∞ errors and max |Δu|/U of u_x on the
    mid-height plane (fluid nodes only)."""
    from pathlib import Path

    ref = tensors.read_reference_csv(Path(data_dir) / "openlb_reference_sphere_force.csv")
    u, area = SPHERE_CASE["u"], math.pi * SPHERE_CASE["diameter"] ** 2 / 4
    cd_ref_t = drag_coefficient(ref["Fx_lattice"], u, area)
    cd_ref = cd_ref_t[ref["step"] >= 0.6 * ref["step"].max()].mean().item()
    cd_ours = cd[int(0.6 * len(cd)) :].mean().item()
    plane = tensors.read_reference_csv(Path(data_dir) / "openlb_reference_sphere_midplane.csv")
    ux_ref = plane["ux"].reshape(SPHERE_CASE["shape"][0], SPHERE_CASE["shape"][1])
    fluid = ux_ref != 0
    diff = (ux_plane.double().cpu() - ux_ref)[fluid]
    return {
        "cd": cd_ours, "cd_ref": cd_ref, "cd_rel": cd_ours / cd_ref - 1,
        "l2": (diff.norm() / ux_ref[fluid].norm()).item(),
        "linf": (diff.abs().max() / ux_ref[fluid].abs().max()).item(),
        "max_du": diff.abs().max().item() / u, "ux_ref": ux_ref, "cd_ref_t": cd_ref_t, "cd_ref_steps": ref["step"],
    }  # fmt: skip


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/lbm_wind_tunnel/plot.py
"""Left: the wind tunnel in 3D — the STL body as a shaded surface, streamlines from an upstream rake and tracer
particles, all coloured by speed, over a floor grid of the lattice. Right: speed in the mid-span plane (the wake) and
the force coefficients C_D(t), C_L(t); in between, the validation residual of this solver against OpenLB on the
sphere case, computed live in the same run."""

import base64
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
from PIL import Image

from problems._common.plotting import export_plot_data, frame_indices
from problems._common.progress import report

SPEED = Normalize(0, 1.3)  # |u| / U, shared by every speed-coloured element and every frame
CMAP = plt.get_cmap("turbo")
RESIDUAL_SCALE = 0.05  # residual maps: fixed symmetric scale ±5 % of U


def shaded_faces(tri: np.ndarray) -> np.ndarray:
    """RGBA per triangle: light blue, darker where the face turns away from a light above the inlet."""
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-12
    light = np.array([-0.5, -0.4, 0.8]) / np.linalg.norm([-0.5, -0.4, 0.8])
    shade = 0.45 + 0.55 * np.clip(n @ light, 0, 1)
    return np.concatenate([np.array([0.58, 0.77, 0.99]) * shade[:, None], np.ones((len(tri), 1))], 1)


def save_gif_shared_palette(fig, update, frames: list[int], path: Path, fps: int = 10) -> Path:
    """Render every frame, then quantise them all with ONE palette built from a strip of frames, so a given speed
    keeps exactly the same colour throughout (Pillow otherwise picks a new palette per frame and colours drift)."""
    fig.set_dpi(80)
    shots = []
    for i, k in enumerate(frames):
        update(k)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)  # numbers behind the final frame (Results page)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for j, im in enumerate(sample):
        strip.paste(im, (0, j * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    colours = np.array(palette.getpalette()[: 3 * 255]).reshape(-1, 3)
    colours[np.argmin(((colours - 255) ** 2).sum(1))] = 255  # keep the background pure white
    palette.putpalette(colours.reshape(-1).tolist())
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


def segments(line: np.ndarray, speed: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Streamlines (S, K, 3) → line segments (M, 2, 3) and their speeds (M,), dropping the stopped (NaN) parts."""
    seg = np.stack([line[:, :-1], line[:, 1:]], 2).reshape(-1, 2, 3)
    val = 0.5 * (speed[:, :-1] + speed[:, 1:]).reshape(-1)
    ok = np.isfinite(seg).all((1, 2)) & np.isfinite(val)
    return seg[ok], val[ok]


def body_length_ticks(set_ticks, set_labels, n: int, step: float, unit: float):
    """Ticks every `step` lattice cells, labelled in body lengths (unit = L in cells)."""
    ticks = np.arange(0, n, step)
    set_ticks(ticks)
    set_labels([f"{v / unit:g}" for v in ticks])


def residual_panel(ax, fig, cax_rect, residual: np.ndarray, extent, title: str, text: str):
    """Reusable validation panel: (ours − reference) / U as a diverging map with a fixed symmetric scale that covers
    every value (at least ±5 %), the error numbers printed on it, and a labelled colour bar."""
    scale = max(RESIDUAL_SCALE, 1.1 * float(np.nanmax(np.abs(residual))))
    image = ax.imshow(residual.T, origin="lower", cmap=plt.get_cmap("RdBu_r", 11), vmin=-scale, vmax=scale,
                      extent=extent, interpolation="nearest", aspect="equal")  # fmt: skip
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_xlabel("mid-height plane $x$–$y$ of the sphere case (lattice nodes)", fontsize=7, labelpad=1)
    ax.text(0.02, 0.05, text, transform=ax.transAxes, fontsize=7, color="#111827",
            bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none", "pad": 1.5})  # fmt: skip
    bar = fig.colorbar(image, cax=fig.add_axes(cax_rect), format=lambda v, _: f"{v:+.0%}")
    bar.ax.tick_params(labelsize=6)
    bar.set_label("$\\Delta u_x/U$ (dimensionless)", fontsize=7)
    return image


def animate(result: dict, coeffs: torch.Tensor, tri: torch.Tensor, solid: torch.Tensor, u_in: float, body: str,
            length: float, path: Path, validation: dict | None = None) -> Path:  # fmt: skip
    tracks = result["tracks"].cpu().numpy()  # (F, P, 3)
    speeds = result["speeds"].cpu().numpy() / u_in  # (F, P)
    lines = result["lines"].cpu().numpy()  # (F, S, K, 3)
    line_speeds = result["line_speeds"].cpu().numpy() / u_in
    slices = result["slices"].cpu().numpy() / u_in  # (F, Nx, Nz), NaN inside the body
    saved = result["steps"].cpu().numpy()
    c = coeffs.cpu().numpy()
    tri = tri.cpu().numpy()
    nx, ny, nz = solid.shape

    fig = plt.figure(figsize=(9.6, 5.8), facecolor="white")
    ax3 = fig.add_axes((0.04, 0.02, 0.52, 0.86), projection="3d")
    ax_s = fig.add_axes((0.6, 0.68, 0.3, 0.22))
    ax_v = fig.add_axes((0.6, 0.38, 0.3, 0.22))
    ax_c = fig.add_axes((0.64, 0.08, 0.27, 0.2))

    # 3D tunnel: equal aspect over the whole lattice, grid panes, ticks in body lengths, floor grid every 2 cells
    ax3.set_xlim(0, nx - 1)
    ax3.set_ylim(0, ny - 1)
    ax3.set_zlim(0, nz - 1)
    ax3.set_box_aspect((nx - 1, ny - 1, nz - 1), zoom=0.95)
    for axis in (ax3.xaxis, ax3.yaxis, ax3.zaxis):
        axis.pane.set_facecolor((0.96, 0.97, 0.98, 1.0))
        axis.pane.set_edgecolor("#d1d5db")
        axis._axinfo["grid"].update(color="#d1d5db", linewidth=0.5)
    body_length_ticks(ax3.set_xticks, ax3.set_xticklabels, nx, length, length)
    body_length_ticks(ax3.set_yticks, ax3.set_yticklabels, ny, length if ny > 3 * length else length / 2, length)
    body_length_ticks(ax3.set_zticks, ax3.set_zticklabels, nz, length if nz > 3 * length else length / 2, length)
    ax3.tick_params(labelsize=7, pad=0)
    ax3.set_xlabel("$x/L$ (dimensionless)", fontsize=7, labelpad=-2)
    ax3.set_ylabel("$y/L$ (dimensionless)", fontsize=7, labelpad=-2)
    ax3.set_zlabel("$z/L$ (dimensionless)", fontsize=7, labelpad=-2)
    floor = [[(x, 0, 0), (x, ny - 1, 0)] for x in range(0, nx, 2)]
    floor += [[(0, y, 0), (nx - 1, y, 0)] for y in range(0, ny, 2)]
    ax3.add_collection3d(Line3DCollection(floor, colors="#9ca3af", linewidths=0.3, alpha=0.6))
    ax3.add_collection3d(Poly3DCollection(tri, facecolors=shaded_faces(tri), edgecolors="none"))
    ax3.quiver(0, ny / 2, nz * 0.6, 8, 0, 0, color="#2563eb", linewidth=1.5, arrow_length_ratio=0.4)
    ax3.text(0, ny / 2, nz * 0.6 + 3, "inflow", color="#1e40af", fontsize=8)
    seg0, val0 = segments(lines[0], line_speeds[0])
    stream = Line3DCollection(seg0, cmap=CMAP, norm=SPEED, linewidths=0.6, alpha=0.4)  # see-through bundle
    stream.set_array(val0)
    ax3.add_collection3d(stream, autolim=False)
    dots = ax3.scatter(*tracks[0].T, c=speeds[0], cmap=CMAP, norm=SPEED, s=1.0, depthshade=False, linewidths=0,
                       alpha=0.7)  # fmt: skip
    title3 = fig.text(0.02, 0.95, " ", fontsize=10)
    fig.text(0.02, 0.915, "streamlines and tracers coloured by speed $|u|/U$ (dimensionless, same scale as the slice)",
             fontsize=7, color="#6b7280")  # fmt: skip

    # mid-span plane: speed |u| / U, lattice grid every 2 cells, equal aspect
    smap = plt.get_cmap("turbo", 13).with_extremes(bad="#9ca3af")
    image = ax_s.imshow(slices[0].T, origin="lower", cmap=smap, norm=SPEED, aspect="equal",
                        extent=(-0.5, nx - 0.5, -0.5, nz - 0.5), interpolation="nearest")  # fmt: skip
    ax_s.set_title("speed $|u|/U$, mid-span plane", fontsize=8)
    ax_s.grid(False)
    ax_s.set_xticks(np.arange(-0.5, nx, 2), minor=True)
    ax_s.set_yticks(np.arange(-0.5, nz, 2), minor=True)
    ax_s.grid(True, which="minor", color="white", linewidth=0.25, alpha=0.5)
    ax_s.tick_params(which="minor", length=0)
    body_length_ticks(ax_s.set_xticks, ax_s.set_xticklabels, nx, length, length)
    body_length_ticks(ax_s.set_yticks, ax_s.set_yticklabels, nz, length / 2, length)
    ax_s.tick_params(labelsize=6)
    ax_s.set_xlabel("$x/L$ (dimensionless; white grid = 2 lattice cells)", fontsize=7, labelpad=1)
    ax_s.set_ylabel("$z/L$", fontsize=7, labelpad=1)
    bar = fig.colorbar(image, cax=fig.add_axes((0.92, 0.68, 0.01, 0.22)))
    bar.ax.tick_params(labelsize=6)
    bar.set_label("speed $|u|/U$ (dimensionless)", fontsize=7)

    # validation: tensor solver − OpenLB on the sphere case, computed live in this run
    if validation is None:
        ax_v.set_axis_off()
    else:
        ux_ref = validation["ux_ref"].numpy()
        res = np.where(ux_ref != 0, (validation["ux_ours"].numpy() - ux_ref) / validation["u"], np.nan)
        text = (f"$\\Delta C_D$ = {validation['cd_rel']:+.1%}   rel. $L_2$ = {validation['l2']:.1%}   "
                f"$L_\\infty$ = {validation['linf']:.1%}")  # fmt: skip
        extent = (-0.5, res.shape[0] - 0.5, -0.5, res.shape[1] - 0.5)
        residual_panel(ax_v, fig, (0.92, 0.38, 0.01, 0.22), res, extent,
                       "tensor solver − OpenLB (sphere, Re = 20): $\\Delta u_x/U$", text)  # fmt: skip

    # force coefficients: fixed limits from the whole history (10 % margin), so nothing is clipped in any frame
    t = np.arange(len(c))
    curves = [c[:, 0], c[:, 2]]
    if body == "sphere_openlb" and validation is not None:
        curves.append(validation["cd_ref_t"].numpy()[1:])  # step 0 of OpenLB is its start-up spike
    lo, hi = min(float(v.min()) for v in curves), max(float(v.max()) for v in curves)
    pad = 0.1 * (hi - lo) if hi > lo else 0.1
    ax_c.set_xlim(0, len(c))
    ax_c.set_ylim(lo - pad, hi + pad)
    ax_c.set_xlabel("lattice time step (steps)", fontsize=7, labelpad=1)
    ax_c.set_ylabel("force coefficient\n$C_D$, $C_L$ (dimensionless)", fontsize=7)
    ax_c.tick_params(labelsize=6)
    drag = ax_c.plot([], [], color="#dc2626", lw=1.2, label="$C_D$")[0]
    lift = ax_c.plot([], [], color="#2563eb", lw=1.0, label="$C_L$")[0]
    if body == "sphere_openlb" and validation is not None:
        ax_c.plot(validation["cd_ref_steps"].numpy()[1:], validation["cd_ref_t"].numpy()[1:], color="#111827", lw=0.7,
                  ls="--", alpha=0.7, label="$C_D$ OpenLB")  # fmt: skip
    mean = ax_c.axhline(np.nan, color="#dc2626", lw=0.8, ls=":")
    ax_c.legend(loc="upper right", fontsize=6, ncol=3)

    frames = frame_indices(len(saved), 48)

    def update(k: int):
        frac = frames.index(k) / max(1, len(frames) - 1)
        ax3.view_init(elev=35, azim=-135 + 20 * (frac - 0.5))  # inlet bottom-left, wake towards top-right, ±10°
        seg, val = segments(lines[k], line_speeds[k])
        stream.set_segments(seg)
        stream.set_array(val)
        dots._offsets3d = tuple(tracks[k].T)
        dots.set_array(speeds[k])
        image.set_data(slices[k].T)
        n = int(saved[k])
        drag.set_data(t[: n + 1], c[: n + 1, 0])
        lift.set_data(t[: n + 1], c[: n + 1, 2])
        cd, cl = c[n // 2 : n + 1, 0].mean(), c[n // 2 : n + 1, 2].mean()
        mean.set_ydata([cd] * 2)
        title3.set_text(f"{body}   step {n}   $C_D \\approx$ {cd:.2f},  $C_L \\approx$ {cl:+.2f}")

    return save_gif_shared_palette(fig, update, frames, path, fps=10)


def _vtk_array(values: np.ndarray, name: str, dtype: str, components: int = 1, unit: str | None = None) -> str:
    """One <DataArray> in VTK's binary XML encoding: base64 of (UInt32 byte count + zlib-compressed block)."""
    raw = np.ascontiguousarray(values, dtype={"Float32": "<f4", "Int32": "<i4"}[dtype]).tobytes()
    packed = zlib.compress(raw, 6)
    header = np.array([1, len(raw), len(raw), len(packed)], dtype="<u4").tobytes()  # 1 block, sizes
    data = base64.b64encode(header).decode() + base64.b64encode(packed).decode()
    extra = f' unit="{unit}"' if unit else ""
    return (f'<DataArray type="{dtype}" Name="{name}" NumberOfComponents="{components}" format="binary"{extra}>'
            f"{data}</DataArray>")  # fmt: skip


def write_vtp(path: Path, points: np.ndarray, cells: list | None = None, kind: str = "Polys",
              point_data: dict | None = None) -> Path:  # fmt: skip
    """Write VTK PolyData (.vtp, XML, zlib-compressed binary). points (P, 3); cells: list of index arrays (one per
    polygon / polyline / vertex), stored as Int32 connectivity + offsets under <Polys>, <Lines> or <Verts>;
    point_data: {name: (values (P,) or (P, k), unit)}."""
    cells = cells if cells is not None else [np.arange(len(points))]
    conn = np.concatenate([np.asarray(c) for c in cells]) if cells else np.zeros(0)
    offsets = np.cumsum([len(c) for c in cells]) if cells else np.zeros(0)
    counts = {k: 0 for k in ("Verts", "Lines", "Polys")}
    counts[kind] = len(cells)
    pd = "".join(
        _vtk_array(v, name, "Float32", 1 if np.ndim(v) == 1 else np.shape(v)[1], unit)
        for name, (v, unit) in (point_data or {}).items()
    )
    xml = (
        '<?xml version="1.0"?>\n<VTKFile type="PolyData" version="1.0" byte_order="LittleEndian" '
        'header_type="UInt32" compressor="vtkZLibDataCompressor"><PolyData>'
        f'<Piece NumberOfPoints="{len(points)}" NumberOfVerts="{counts["Verts"]}" NumberOfLines="{counts["Lines"]}" '
        f'NumberOfStrips="0" NumberOfPolys="{counts["Polys"]}">'
        f"<PointData>{pd}</PointData><Points>{_vtk_array(points, 'Points', 'Float32', 3)}</Points>"
        f"<{kind}>{_vtk_array(conn, 'connectivity', 'Int32')}{_vtk_array(offsets, 'offsets', 'Int32')}</{kind}>"
        "</Piece></PolyData></VTKFile>\n"
    )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(xml, encoding="utf-8")
    return path


def write_pvd(path: Path, entries: list[tuple[float, str, str]]) -> Path:
    """ParaView collection: entries (time, layer name, file relative to the .pvd); each layer is one 'part'."""
    layers = list(dict.fromkeys(name for _, name, _ in entries))
    rows = "".join(
        f'<DataSet timestep="{t:.6g}" group="" part="{layers.index(name)}" name="{name}" file="{file}"/>'
        for t, name, file in entries
    )
    path = Path(path)
    path.write_text(f'<?xml version="1.0"?>\n<VTKFile type="Collection" version="1.0" byte_order="LittleEndian">'
                    f"<Collection>{rows}</Collection></VTKFile>\n", encoding="ascii")  # fmt: skip
    return path


def read_vtp(path: Path) -> dict:
    """Parse a .vtp written by write_vtp back: counts, point coordinates and named point arrays (for the tests)."""
    import xml.etree.ElementTree as ET

    def decode(node) -> np.ndarray:
        text = node.text.strip()
        head = np.frombuffer(base64.b64decode(text[:24]), dtype="<u4")  # 4 UInt32 = 16 bytes = 24 base64 chars
        raw = zlib.decompress(base64.b64decode(text[24:]))
        assert len(raw) == head[1]
        return np.frombuffer(raw, dtype={"Float32": "<f4", "Int32": "<i4"}[node.get("type")])

    piece = ET.parse(path).getroot().find("PolyData/Piece")
    out = {k: int(piece.get(f"NumberOf{k}")) for k in ("Points", "Verts", "Lines", "Polys")}
    out["points"] = decode(piece.find("Points/DataArray")).reshape(-1, 3)
    out["arrays"] = {a.get("Name"): decode(a) for a in piece.find("PointData")}
    for kind in ("Verts", "Lines", "Polys"):
        node = piece.find(kind)
        if node is not None:
            arrays = {a.get("Name"): decode(a) for a in node}
            out["connectivity"], out["offsets"] = arrays["connectivity"], arrays["offsets"]
    return out


def export_vtk(result: dict, tri: torch.Tensor, solid: torch.Tensor, u_in: float, length: float, out_dir: Path,
               n_steps: int = 24) -> Path:  # fmt: skip
    """Animated VTK for the web viewer and ParaView, in results/ (the .pvd and its .vtp files): wind_tunnel.pvd with
    surface (STL triangles, point array Cp, static), streamlines (polylines, speed |u|/U), particles (vertices, speed
    and age) and wake (iso-surface |u|/U = 0.7 of the final field, vorticity |ω| L/U, static). Coordinates in body
    lengths, x downstream, z up; time in lattice steps."""
    from . import tensors
    from .solve import interpolate, macroscopic

    res = Path(out_dir) / "results"
    e, _, _ = tensors.lattice(result["f"].device, result["f"].dtype)
    rho, u = macroscopic(result["f"], e)
    entries = []

    # surface: C_p = (p − p∞)/(½ρU²), p = ρ/3, sampled one lattice cell outside every face (each face keeps 3 vertices)
    cp_field = (rho / 3 - rho[1].mean() / 3) / (0.5 * u_in**2)
    v = tri.reshape(-1, 3)
    normal = torch.linalg.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    probe = v + (normal / normal.norm(dim=1, keepdim=True).clamp_min(1e-12)).repeat_interleave(3, 0)
    cp = interpolate(cp_field[None], probe)[:, 0].cpu().numpy()
    write_vtp(res / "surface.vtp", (v / length).cpu().numpy(), list(np.arange(len(v)).reshape(-1, 3)), "Polys",
              {"Cp": (cp, "dimensionless")})  # fmt: skip

    # wake: iso-surface of the final speed field, coloured by vorticity magnitude
    speed = (u.norm(dim=0) / u_in).masked_fill(solid, 0.0)
    grad = [torch.gradient(u[a], dim=(0, 1, 2)) for a in range(3)]  # grad[a][b] = ∂u_a/∂x_b
    omega = torch.stack([grad[2][1] - grad[1][2], grad[0][2] - grad[2][0], grad[1][0] - grad[0][1]])
    wmag = (omega.norm(dim=0) * length / u_in).masked_fill(solid, 0.0)
    verts, faces = tensors.surface_nets(-speed, -0.7)  # inside = slower than 0.7 U
    if len(faces):
        at = verts.to(u.dtype)
        wake_data = {
            "vorticity": (interpolate(wmag[None], at)[:, 0].cpu().numpy(), "dimensionless (|ω| L/U)"),
            "speed": (interpolate(speed[None], at)[:, 0].cpu().numpy(), "dimensionless (|u|/U)"),
        }
        write_vtp(res / "wake.vtp", (verts / length).cpu().numpy(), list(faces.cpu().numpy()), "Polys", wake_data)

    saved = result["steps"].cpu().numpy()
    frames = frame_indices(len(saved), n_steps)
    for j, k in enumerate(frames):
        t = float(saved[k])
        entries += [(t, "surface", "surface.vtp")]
        if len(faces):
            entries += [(t, "wake", "wake.vtp")]
        lines = result["lines"][k].cpu().numpy()  # (S, K, 3), NaN after a line stops
        line_speed = result["line_speeds"][k].cpu().numpy() / u_in
        pts, val, cells, start = [], [], [], 0
        for line, sp in zip(lines, line_speed):  # loop over the 45 rake lines to pack them, not over nodes
            ok = np.isfinite(line).all(1) & np.isfinite(sp)
            m = int(np.argmin(ok)) if not ok.all() else len(ok)  # keep the valid head of each line
            if m >= 2:
                pts.append(line[:m])
                val.append(sp[:m])
                cells.append(np.arange(start, start + m))
                start += m
        write_vtp(res / f"streamlines_{j:03d}.vtp", np.concatenate(pts) / length, cells, "Lines",
                  {"speed": (np.concatenate(val), "dimensionless (|u|/U)")})  # fmt: skip
        p = result["tracks"][k].cpu().numpy()
        write_vtp(res / f"particles_{j:03d}.vtp", p / length, list(np.arange(len(p))[:, None]), "Verts",
                  {"speed": (result["speeds"][k].cpu().numpy() / u_in, "dimensionless (|u|/U)"),
                   "age": (result["ages"][k].cpu().numpy(), "lattice steps")})  # fmt: skip
        entries += [(t, "streamlines", f"streamlines_{j:03d}.vtp"),
                    (t, "particles", f"particles_{j:03d}.vtp")]  # fmt: skip
    return write_pvd(res / "wind_tunnel.pvd", entries)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'body': 'airplane',
    're': 1000.0,
    'collision': 'trt',
    'les': 'on',
    'pitch': 8.0,
    'validate': 'on',
    'steps': 1200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.lbm_wind_tunnel")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.lbm_wind_tunnel.main")
problem_dir = Path("problems/fluids/lbm_wind_tunnel")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
